# Bidirectional LSTM
Notebook: `d21_s3_a2.ipynb`


In [1]:
import numpy as np
rng = np.random.RandomState(2)
D, H = 3, 4
Wxf = rng.randn(4*H, D)*0.5; Whf = rng.randn(4*H, H)*0.5; bf = rng.randn(4*H)*0.3
Wxb = rng.randn(4*H, D)*0.5; Whb = rng.randn(4*H, H)*0.5; bb = rng.randn(4*H)*0.3
X = rng.randn(5, D)


In [2]:
def sigmoid(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    exp_x = np.exp(x[~pos])
    out[~pos] = exp_x / (1.0 + exp_x)
    return out

def lstm_cell_step(x_t, h_prev, c_prev, Wx, Wh, b):
    H = h_prev.shape[0]
    z = Wx @ x_t + Wh @ h_prev + b
    i = sigmoid(z[0:H])
    f = sigmoid(z[H:2*H])
    g = np.tanh(z[2*H:3*H])
    o = sigmoid(z[3*H:4*H])
    c_t = f * c_prev + i * g
    h_t = o * np.tanh(c_t)
    return h_t, c_t

def lstm_run(X, Wx, Wh, b):
    H = Wh.shape[1]
    h = np.zeros(H)
    c = np.zeros(H)
    hidden_states = []
    for x_t in X:
        h, c = lstm_cell_step(x_t, h, c, Wx, Wh, b)
        hidden_states.append(h)
    return np.array(hidden_states)

def bilstm_forward(X, Wxf, Whf, bf, Wxb, Whb, bb):
    Hf = lstm_run(X, Wxf, Whf, bf)
    Hb = lstm_run(X[::-1], Wxb, Whb, bb)[::-1]
    return np.concatenate([Hf, Hb], axis=1)


In [3]:
O = bilstm_forward(X, Wxf, Whf, bf, Wxb, Whb, bb)
print(O.shape)
print(np.allclose(O[:, :4], lstm_run(X, Wxf, Whf, bf)))
Hb = lstm_run(X[::-1], Wxb, Whb, bb)[::-1]
print(np.allclose(O[:, 4:], Hb))


(5, 8)
True
True


In [4]:
# Additional autograder-style checks
assert np.allclose(sigmoid(np.array([-1000., 0., 2.])), [0., 0.5, 0.8807970779778823])
assert lstm_run(X, Wxf, Whf, bf).shape == (5, 4)
assert O.shape == (5, 8)
backward_reversed_run = lstm_run(X[::-1], Wxb, Whb, bb)
assert np.allclose(O[0, 4:], backward_reversed_run[-1])
O_rev_swap = bilstm_forward(X[::-1], Wxb, Whb, bb, Wxf, Whf, bf)
expected = np.concatenate([O[::-1, 4:], O[::-1, :4]], axis=1)
assert np.allclose(O_rev_swap, expected)
print('All additional checks passed.')


All additional checks passed.
